In [1]:
import pandas as pd 
import numpy as np 
from alphagenome import colab_utils
from alphagenome.data import genome
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers

/opt/anaconda3/envs/opensplicerna/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")
api_key = os.environ["ALPHA_GENOME_API_KEY"] #api key I generated from google deepmind

In [14]:
from alphagenome.models import dna_client

dna_model = dna_client.create(api_key)

In [15]:
splicing_scorers = [
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITES'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITE_USAGE'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_JUNCTIONS'],
]

for scorer in splicing_scorers:
  print(f'{scorer.name} (signed={scorer.is_signed})')

GeneMaskSplicingScorer(requested_output=SPLICE_SITES, width=None) (signed=False)
GeneMaskSplicingScorer(requested_output=SPLICE_SITE_USAGE, width=None) (signed=False)
SpliceJunctionScorer() (signed=False)


In [16]:
# Define a variant near a splice site in the BRCA2 gene.
variant = genome.Variant(
    chromosome='chr13',
    position=32316462,
    reference_bases='T',
    alternate_bases='G',
)

# Create a 1MB interval centered on the variant.
interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

# Score the variant using the splicing scorers.
scores = dna_model.score_variant(
    interval=interval,
    variant=variant,
    variant_scorers=splicing_scorers,
    organism=dna_client.Organism.HOMO_SAPIENS,
)

# View the tidy scores.
df_scores = variant_scorers.tidy_scores([scores])
df_scores

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,...,track_strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,gtex_tissue,data_source,raw_score,quantile_score
0,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,None,None,SPLICE_SITES,GeneMaskSplicingScorer(requested_output=SPLICE...,...,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.011108,0.765994
1,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,None,None,SPLICE_SITES,GeneMaskSplicingScorer(requested_output=SPLICE...,...,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003906,0.045249
2,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,...,.,polyA plus RNA-seq,CL:0000047,neuronal stem cell,in_vitro_differentiated_cells,embryonic,,encode,0.007812,0.525389
3,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,...,.,total RNA-seq,CL:0000062,osteoblast,primary_cell,adult,,encode,0.011719,0.946842
4,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,...,.,polyA plus RNA-seq,CL:0000084,T-cell,primary_cell,adult,,encode,0.011719,0.958945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2199,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,32315667,32319076,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,,encode,0.052551,0.985728
2200,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,32316527,32325075,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,,encode,0.050415,0.984743
2201,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,32316391,32319076,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,,encode,0.045563,0.982567
2202,chr13:32316462:T>G,chr13:31792174-32840750:.,ENSG00000139618,BRCA2,protein_coding,+,32316391,32316421,SPLICE_JUNCTIONS,SpliceJunctionScorer(),...,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,,encode,0.014336,0.753630
